# Cost of simpler nf-core/mag configurations

The benchmark ran every assembler and binner at once. Most users will run far less than that,
so this notebook costs out two minimal configurations by selecting, from the same traces, only
the tasks each one would have launched. Reported per sample: jobs, CPU core-hours, work-directory
size, and peak memory.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import polars as pl

from mag_bench.build import prepare
from mag_bench.configs import Config, estimate, estimate_by_stage

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(120)

frames = prepare("../data")

## Sample configurations

One short-read assembly with a classic and a modern binner, one long-read assembly with a single
binner. Both classify bins with GTDB-Tk and keep one bin QC tool each. The stage list is explicit,
so what is left out is auditable.


In [2]:
SHORT_READ = Config(
    name="short-read (MEGAHIT)",
    assembler="MEGAHIT",
    binners=("MetaBAT2", "SemiBin2"),
    stages=(
        "Short-read preprocessing",
        "Short-read assembly",
        "Assembly QC (QUAST)",
        "Read mapping + depth",
        "Binning",
        "Bin QC (CheckM2)",
        "Taxonomic classification (GTDB-Tk)",
        "Reporting",
    ),
)

LONG_READ = Config(
    name="long-read (metaMDBG)",
    assembler="METAMDBG_ASM",
    binners=("MetaBinner",),
    stages=(
        "Long-read preprocessing",
        "Long-read assembly",
        "Assembly QC (QUAST)",
        "Read mapping + depth",
        "Binning",
        "Bin QC (BUSCO)",
        "Taxonomic classification (GTDB-Tk)",
        "Reporting",
    ),
)

## Cost per sample

Peak memory is the largest single task, not a per-sample sum: it is the memory a node must have.


In [3]:
estimate(frames["trace"], frames["samples"], [SHORT_READ, LONG_READ])

config,dataset,jobs_per_sample,cpu_hours_per_sample,workdir_gb_per_sample,peak_rss_gb
str,str,f64,f64,f64,f64
"""long-read (metaMDBG)""","""maghini""",20.5,27.0,39.8,205.4
"""long-read (metaMDBG)""","""zymo""",25.0,45.4,74.6,204.1
"""short-read (MEGAHIT)""","""maghini""",22.6,41.8,20.9,205.3
"""short-read (MEGAHIT)""","""zymo""",28.0,56.3,28.1,151.9


## Where the cost goes

Both datasets pooled, one row per analysis stage.


In [4]:
estimate_by_stage(frames["trace"], frames["samples"], SHORT_READ)

stage,jobs_per_sample,cpu_hours_per_sample,workdir_gb_per_sample,peak_rss_gb
enum,f64,f64,f64,f64
"""Short-read assembly""",2.0,18.3,2.3,24.4
"""Short-read preprocessing""",4.2,13.3,13.0,2.9
"""Read mapping + depth""",6.0,7.8,4.9,8.2
"""Bin QC (CheckM2)""",2.2,2.1,0.2,17.9
"""Binning""",5.0,0.7,0.4,4.2
"""Taxonomic classification (GTDB…",2.2,0.7,0.5,205.3
"""Assembly QC (QUAST)""",1.0,0.1,0.0,1.2
"""Reporting""",0.5,0.0,0.1,12.0


In [5]:
estimate_by_stage(frames["trace"], frames["samples"], LONG_READ)

stage,jobs_per_sample,cpu_hours_per_sample,workdir_gb_per_sample,peak_rss_gb
enum,f64,f64,f64,f64
"""Long-read assembly""",2.0,16.8,0.4,9.9
"""Bin QC (BUSCO)""",1.2,4.3,0.7,10.9
"""Binning""",6.0,3.1,17.4,4.1
"""Long-read preprocessing""",4.0,2.0,15.9,57.9
"""Read mapping + depth""",5.0,1.6,8.0,14.3
"""Taxonomic classification (GTDB…",1.2,0.7,0.3,205.4
"""Assembly QC (QUAST)""",1.0,0.1,0.0,1.0
"""Reporting""",0.5,0.0,0.1,12.0


## What these numbers assume

- **The estimates are subsets of a run that did more.** Downstream tasks are attributed to the
  assembly named in their task tag, so a configuration is costed from the tasks that actually ran
  on that assembly. Nothing is extrapolated.
- **DAS Tool cannot be estimated this way and is excluded.** With `postbinning_input: both` it
  refines the union of every binner that ran, so its cost with six binners does not transfer to a
  configuration with one or two.
- **The long-read assembly was polished with short reads.** `pypolca` is a short-read polisher, so
  a long-read-only run cannot do it. Binning and QC here ran on the polished metaMDBG assembly;
  its size is nearly identical unpolished, but this is an assumption rather than a measurement.
- **SemiBin2 used the pretrained model** (`semibin_environment: global`). Self-supervised training
  costs far more, so its number is not transferable to that mode.
- **Other pipeline defaults are switched off here**, including ALE, bin-level QUAST, GUNC,
  Prodigal/Prokka and CAT. A run left on defaults costs more than these tables show. Prokka in
  particular dominates job counts, at thousands of tasks per sample.
- **Once-per-run tasks are divided by the sample count.** The PhiX index, MultiQC and the summary
  concatenations do not scale with samples, so they inflate the per-sample figures of a small run.
- **Runtime is deliberately absent.** Wall-clock is set by the critical path and by cluster
  contention, not by the sum of task times. Dropping binners frees slots rather than shortening the
  chain, so it can cut core-hours several-fold while barely moving elapsed time.
- **Per-assembly helper tasks are included as they ran.** `CONVERT_DEPTHS` prepares the depth format
  that MaxBin2 and CONCOCT need, and the pipeline may skip it when only binners that do not need it
  are enabled. Keeping it makes the short-read estimate marginally conservative.
